<a href="https://colab.research.google.com/github/JaiJanarious/steam-gaming-for-investors/blob/main/Gaming_For_Investors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Gaming for investors App.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Mounted at /content/drive
✅ Loaded 1000 games from: /content/drive/MyDrive/MIST 4600/


In [76]:
import requests
import pandas as pd

STEAMSPY_URL = "https://steamspy.com/api.php?request=top100forever"
steamspy_response = requests.get(STEAMSPY_URL).json()  # dict: appid -> game info
top_games = pd.DataFrame(steamspy_response.values())   # one row per game

# SteamSpy gives owners as a text range like "50,000,000 .. 100,000,000".
# Split it into low / high and use the midpoint as a single number.
owner_range = top_games["owners"].str.replace(",", "").str.split(" .. ", expand=True)
top_games["owners_midpoint"] = (owner_range[0].astype(int) + owner_range[1].astype(int)) / 2

# Popularity score: scale each metric to 0-1 (biggest game = 1), then average the three
popularity_columns = ["ccu", "average_forever", "owners_midpoint"]
scaled_metrics = top_games[popularity_columns] / top_games[popularity_columns].max()
top_games["popularity_score"] = scaled_metrics.mean(axis=1)

# 2. Group the top games by developer

# developer name -> {"game_count": ..., "score_sum": ...}
developer_totals = {}

for _, game in top_games.iterrows():
    # setdefault creates the developer's entry the first time we see them
    totals = developer_totals.setdefault(game["developer"], {"game_count": 0, "score_sum": 0.0})
    totals["game_count"] += 1
    totals["score_sum"] += game["popularity_score"]

# Keep only developers with 2+ games in the top 100, and work out their average score.
# developer name -> {"top100_game_count": ..., "avg_popularity_score": ...}
developer_stats = {
    developer: {
        "top100_game_count": totals["game_count"],
        "avg_popularity_score": round(totals["score_sum"] / totals["game_count"], 3),
    }
    for developer, totals in developer_totals.items()
    if totals["game_count"] >= 2
}
step1_table = pd.DataFrame.from_dict(developer_stats, orient="index")
step1_table.index.name = "developer"
step1_table.sort_values("avg_popularity_score", ascending=False)

,top100_game_count,avg_popularity_score
developer,,
Valve,12,0.198
Facepunch Studios,2,0.166
Bohemia Interactive,2,0.083
CD PROJEKT RED,2,0.073
"Firaxis Games, Aspyr (Mac), Aspyr (Linux)",2,0.057
"FromSoftware, Inc.",3,0.056
"CAPCOM Co., Ltd.",2,0.050
Bethesda Game Studios,2,0.048
Endnight Games Ltd,2,0.034


In [75]:
for developer, stats in developer_stats.items():
    developer_games = games[games["developers"] == developer]  # this developer's games in the CSV
    stats["csv_game_count"] = len(developer_games)

    release_years = sorted(developer_games["release_year"])  # earliest first
    if len(release_years) >= 2:
        # pair each year with the next one and take the difference
        year_gaps = [later - earlier for earlier, later in zip(release_years, release_years[1:])]
        stats["avg_release_gap_years"] = round(sum(year_gaps) / len(year_gaps), 2)
    else:
        stats["avg_release_gap_years"] = None  # need 2+ games to have a gap

# Show the table (fastest releasers first)
step2_table = pd.DataFrame.from_dict(developer_stats, orient="index")
step2_table.index.name = "developer"
step2_table.sort_values("avg_release_gap_years")

,top100_game_count,avg_popularity_score,csv_game_count,avg_release_gap_years
developer,,,,
DICE,2,0.027,5,0.25
"CAPCOM Co., Ltd.",2,0.050,16,0.67
"FromSoftware, Inc.",3,0.056,6,1.80
Bethesda Game Studios,2,0.048,6,2.40
CD PROJEKT RED,2,0.073,3,2.50
Valve,12,0.198,8,2.57
Bohemia Interactive,2,0.083,3,5.00
Endnight Games Ltd,2,0.034,2,6.00
"Firaxis Games, Aspyr (Mac), Aspyr (Linux)",2,0.057,2,6.00


In [78]:
for developer, stats in developer_stats.items():
    # only this developer's games that have at least 1 review
    reviewed_games = games[(games["developers"] == developer) & (games["num_reviews_total"] > 0)]
    review_counts = reviewed_games["num_reviews_total"]

    stats["reviewed_game_count"] = len(reviewed_games)
    stats["total_reviews"] = int(review_counts.sum())

    if len(reviewed_games) > 0:
        # Weighted average of percent positive: games with more reviews count for more
        weighted_sum = (reviewed_games["pct_pos_total"] * review_counts).sum()
        stats["weighted_pct_positive"] = round(weighted_sum / review_counts.sum(), 1)
    else:
        stats["weighted_pct_positive"] = None  # nothing to average

# Show the table (best reviewed first)
step3_table = pd.DataFrame.from_dict(developer_stats, orient="index")
step3_table.index.name = "developer"
step3_table.sort_values("weighted_pct_positive", ascending=False)

,top100_game_count,avg_popularity_score,reviewed_game_count,total_reviews,weighted_pct_positive
developer,,,,,
Endnight Games Ltd,2,0.034,2,739121,92.7
"FromSoftware, Inc.",3,0.056,6,1399311,92.3
Facepunch Studios,2,0.166,2,1978866,91.5
CD PROJEKT RED,2,0.073,3,1507265,90.5
"Firaxis Games, Aspyr (Mac), Aspyr (Linux)",2,0.057,2,373949,89.0
"CAPCOM Co., Ltd.",2,0.050,16,1211791,86.7
Valve,12,0.198,8,12579456,85.4
Bethesda Game Studios,2,0.048,6,868156,84.4
Bohemia Interactive,2,0.083,3,585603,81.6
